In [2]:
# !unzip lib
# !unzip data

In [3]:
from collections import OrderedDict
from scipy.optimize import fmin_l_bfgs_b
from lib.Q3 import (get_loss, 
                    get_gradients, 
                    get_sentences_tags)
import numpy as np
import pandas as pd

In [4]:
emissionParametersdf = pd.read_csv('./data/partial/emissionParameters.csv')
transmissionParametersdf = pd.read_csv('./data/partial/transitionParameters.csv')
transmissionParametersdf = transmissionParametersdf.set_index("feature", inplace=False)
transmissionParamsDict = transmissionParametersdf["weight"].to_dict()
emissionParametersdf = emissionParametersdf.set_index('feature', inplace=False)
emissionParamsDict = emissionParametersdf["weight"].to_dict()


transmissionCountDict=transmissionParametersdf["count[y(i-1),y]"].to_dict()
emissionCountDict = emissionParametersdf["count[y,x]"].to_dict()
tagsdf=pd.concat([transmissionParametersdf['y'],emissionParametersdf['y']])

tags = tagsdf.unique()


# tags
emission_feature_dict=(emissionParametersdf['weight']*emissionParametersdf['count[y,x]']).to_dict()
transition_feature_dict=(transmissionParametersdf['weight']*transmissionParametersdf['count[y(i-1),y]'])

In [5]:
emissionParamsDict.update(transmissionParamsDict)
feature_dict = OrderedDict(emissionParamsDict)

allFeatures = list(feature_dict.keys())
knownFeatures = [feature for feature in allFeatures if not "<unk>" in feature]
FEATURE_VECTOR = knownFeatures
TAGS = np.delete(tags,3)
SENTENCES, SENTENCE_TAGS = get_sentences_tags('./data/partial/train')

In [6]:
'''

FEATURES = list(feature_dict_input.keys())
weights =  list(feature_dict_input.values())
     get_loss(sentences, sentence_tags, tags, feature_dict)
get_gradients(sentences, sentence_tags, feature_dict, tags)

'''
def regularizedLoss(weightVector, 
                    sentences=SENTENCES, 
                    sentence_tags=SENTENCE_TAGS, 
                    tags=TAGS, 
                    featureVector=FEATURE_VECTOR, 
                    eta=0.1):
    if len(weightVector) != len(featureVector):
        raise ValueError("Length of weight Vector doesn't match that of the feature vector")
        
    keys_list = featureVector
    values_list = weightVector
    zip_iterator = zip(keys_list, values_list)
    feature_dict = OrderedDict(zip_iterator)
    
    loss = get_loss(sentences, sentence_tags, tags, feature_dict)
    
    if type(weightVector) != np.ndarray:
        w = np.array(weightVector)
    else:
        w = weightVector
        
    regularization = w.dot(w) * eta
    
    return loss + regularization

def grad(weightVector,
        sentences=SENTENCES, 
        sentence_tags=SENTENCE_TAGS, 
        tags=TAGS,
        featureVector=FEATURE_VECTOR):

    if len(weightVector) != len(featureVector):
        raise ValueError("Length of weight Vector doesn't match that of the feature vector")
        
    keys_list = featureVector
    values_list = weightVector
    zip_iterator = zip(keys_list, values_list)
    feature_dict = OrderedDict(zip_iterator)
    
    
    gradients = get_gradients(sentences, sentence_tags, feature_dict, tags)

    gradVec = OrderedDict()
    for feature in feature_dict.keys():
        gradVec[feature] = gradients.get(feature, -1e9)
    
    gradientVector = np.array(list(gradVec.values()))
    
    return gradientVector

In [7]:
def get_loss_grad(w):
    """
    This function will only be called by "fmin_l_bfgs_b"
    Arg:
        w: weights, numpy array
    Returns:
        loss: loss, float
        grads: gradients, numpy array
    """
    # to be completed by you,
    # based on the modified loss and gradients,
    # with L2 regularization included
    
    loss = regularizedLoss(w)
    grads = grad(w)
    return loss, grads

In [8]:
# len(FEATURE_VECTOR) # was 13651 including <unk> features

In [10]:
def callbackF(w):
    '''
    This function will only be called by "fmin_l_bfgs_b"
    Arg:
        w: weights, numpy array
    '''
    loss = regularizedLoss(w)
    print('Loss:{0:.4f}'.format(loss))

init_w = np.zeros(len(FEATURE_VECTOR))
result = fmin_l_bfgs_b(regularizedLoss, 
                       init_w, 
                       grad,
                       pgtol=0.01, 
                       callback=callbackF)

print(result)

/content/lib/Q3.py:156: RuntimeWarning: divide by zero encountered in log
  beta[i,current_position]=np.log(np.sum(np.exp(feature_score)))


Loss:15736.9719
Loss:9694.8753
Loss:8667.0926
Loss:7812.3029
Loss:7309.1347
Loss:6745.7205
Loss:6023.4069
Loss:5487.2834
Loss:5065.7616
Loss:4888.5838
Loss:4714.0290
Loss:4558.8257
Loss:4303.4114
Loss:4226.7720
Loss:4085.4789
Loss:3988.7444
Loss:3880.4873
Loss:3821.5223
Loss:3759.4729
Loss:3681.5627
Loss:3583.5090
Loss:3541.1361
Loss:3500.4498
Loss:3459.5372
Loss:3429.4040
Loss:3422.2633
Loss:3415.7001
Loss:3409.7010
Loss:3400.6626
Loss:3400.5030
Loss:3399.5286
Loss:3398.3482
Loss:3397.8937
Loss:3397.0020


/content/lib/Q3.py:156: RuntimeWarning: overflow encountered in exp
  beta[i,current_position]=np.log(np.sum(np.exp(feature_score)))
/usr/local/lib/python3.7/dist-packages/numpy/core/fromnumeric.py:87: RuntimeWarning: overflow encountered in reduce
  return ufunc.reduce(obj, axis, dtype, out, **passkwargs)


Loss:3397.0020
(array([ 4.18168133,  3.1889708 ,  6.4716844 , ..., -0.48806967,
       -0.70664686, -0.43154772]), 3397.0020434156927, {'grad': array([ 1.72833896,  2.64464483, 10.03037759, ..., -0.05551681,
        1.8983073 ,  0.06731562]), 'task': b'CONVERGENCE: REL_REDUCTION_OF_F_<=_FACTR*EPSMCH', 'funcalls': 79, 'nit': 35, 'warnflag': 0})


In [11]:
from joblib import dump

dump(dict(zip(FEATURE_VECTOR, result[0])), "q4_final_feature_dict.joblib")

['q4_final_feature_dict.joblib']